In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_order_items = spark.read.table('global_partner.silver.order_items')
df_order_item_options = spark.read.table('global_partner.silver.order_item_options')
df_date_dim = spark.read.table('global_partner.bronze.date_dim')

In [0]:
print(df_order_items.count())
print(df_order_item_options.count())

1.	Customer Lifetime Value (CLV):  
Goal: Estimate how much total revenue a customer will generate over their entire relationship with the business.  

Why it matters: Helps prioritize high-value customers, plan marketing budgets wisely, and improve retention strategies.  
How to do it:
- ●	Use order_items and order_item_options to compute revenue per order.
- ●	Aggregate total spend per customer_id.
- Group CLV values (for tagging):
- ●	High CLV: Top 20% customers
- ●	Medium CLV: Mid 60%
- ●	Low CLV: Bottom 20%


Use order_items and order_item_options to compute revenue per order.

In [0]:
agg_df = df_order_items.groupBy(['order_id']).agg(F.round(F.sum('item_price'),2).alias('order_amount')).orderBy('order_amount',ascending=False)
agg_df.display()

Aggregate total spend per customer_id.

In [0]:
agg_df = df_order_items.groupBy(['user_id']).agg(F.round(F.sum('item_price'),2).alias('order_amount')).orderBy('order_amount',ascending=False)
agg_df.display()

In [0]:
clv_window = Window.orderBy(F.col('order_amount').desc(),F.col('user_id'))

agg_df = (agg_df.withColumn('CLV',F.percent_rank().over(clv_window))
          .withColumn('clv_segment',F.when(F.col('CLV') <= 0.2,'High CLV')
                                .when((F.col('CLV') > 0.2) & (F.col('CLV') <= 0.8),'Medium CLV')
                                .otherwise('Low CLV'))
          )
agg_df.display()

In [0]:
df_order_item_options.display()

In [0]:
df_options = df_order_item_options.withColumn('total_price',(F.col('OPTION_PRICE')*F.col('OPTION_QUANTITY')))
agg_options = df_options.groupBy('ORDER_ID').agg(F.sum('total_price').alias('tot_price'))
agg_options.display()

In [0]:
print(agg_df.count())

In [0]:
df = df_order_items.filter(F.col('ORDER_ID')=='645fdb6ebf57a89e4800829d')
df.display()

In [0]:
df1 = df_order_item_options.filter(F.col('ORDER_ID')=='645fdb6ebf57a89e4800829d')
df1.display()

In [0]:
%sql
select ITEM_PRICE,ITEM_QUANTITY,ORDER_ID,
RESTAURANT_ID,ITEM_NAME,CREATION_TIME_UTC
from global_partner.silver.order_items
order by RESTAURANT_ID,ITEM_NAME,CREATION_TIME_UTC

In [0]:
%sql
SELECT *
FROM global_partner.silver.order_item_options
WHERE order_id in ('5fdbff5f37ab46af60e888e2','5fdc00f0505ee9fe089f9196','5fe354234f5ee982340e3ed8')

In [0]:
df = df_order_items.filter((F.col('ITEM_NAME')=='Greek Salad') & (F.col('RESTAURANT_ID')=='5e7e35ec902ad5ac017b242a'))
df.display()

In [0]:
merged_df = df_order_items.join(df_order_item_options,
                                on=['ORDER_ID','LINEITEM_ID'],
                                how='left')

In [0]:
merged_df.display()

In [0]:
merged_df = merged_df.fillna(0,subset=['option_price','option_quantity'])

In [0]:
merged_df = merged_df.fillna('NA',subset=['option_group_name','option_name'])

In [0]:
df = merged_df.filter(F.col('option_price').isNull())
df.display()

In [0]:
merged_df.withColumn('total_amount')

In [0]:
agg_df = merged_df.groupBy(['order_id','lineitem_id']).agg(F.count('*').alias('cnt')).orderBy('cnt',ascending=False)
agg_df.display()

In [0]:
df = merged_df.filter(F.col('order_id')=='5ed7ab56505ee9030f7b23d0')
df.display()

In [0]:
agg_df = df_order_items.groupBy(['order_id']).agg(F.count('*').alias('cnt')).orderBy('cnt',ascending=False)
agg_df.display()
print(agg_df.count())

Secondary Metrics
2.	Customer Segmentation & Behavior:  
Goal: Group customers based on spending and activity to support campaign targeting.    
Why it matters: Enables personalized offers and engagement.  
How to do it:  

 ●	Use RFM logic based on order_items:
- ○	Recency: Days since last purchase
- ○	Frequency: Number of purchases in last N months
- ○	Monetary: Total spend in last N months. 

●	Segment:
- ○	VIPs: High R, F, M
- ○	New Customers: Low F, high R
- ○	Churn Risk: Low R, low F


In [0]:
df_order_items = df_order_items.withColumn('CREATION_TIME_UTC',F.to_timestamp('CREATION_TIME_UTC'))

In [0]:
df_segment = df_order_items.select('CREATION_TIME_UTC','USER_ID','ORDER_ID','ITEM_PRICE').distinct()
df_segment.display()

In [0]:
snapshot_date = (df_segment.agg(F.max('CREATION_TIME_UTC').alias('snapshot_date')).first()['snapshot_date'])
print(snapshot_date)
print(type(snapshot_date))
df_segment = df_segment.filter(F.col('CREATION_TIME_UTC') <= F.add_months(F.lit(snapshot_date),-12))

In [0]:
df_segment.display()

In [0]:
df_segment_final = (df_segment.groupBy('USER_ID')
              .agg(F.datediff(F.lit(snapshot_date),F.max('CREATION_TIME_UTC')).alias('recency'),
                   F.countDistinct('order_id').alias('frequency'),
                   F.round(F.sum('ITEM_PRICE'),2).alias('Total_amt'))
    )
df_segment_final.display()

In [0]:
recency_window = Window.orderBy(F.col('recency').desc())
freq_window = Window.orderBy(F.col('frequency').desc())
total_amt_window = Window.orderBy(F.col('total_amt').desc())

In [0]:
df_segment_final = (df_segment_final.withColumn('rnk_recency',F.percent_rank().over(recency_window))
                    .withColumn('rnk_frq',F.percent_rank().over(freq_window))
                    .withColumn('rnk_amt',F.percent_rank().over(total_amt_window))
                    .withColumn('segment',F.when(
                        (F.col('rnk_recency') > 0.5) & 
                        (F.col('rnk_frq') > 0.5) & 
                        (F.col('rnk_amt') > 0.5),'VIPs')
                                .when(
                            (F.col('rnk_recency') > 0.5) & 
                        (F.col('rnk_frq') < 0.5),'New Customers')
                            .when(
                            (F.col('rnk_recency') < 0.5) & 
                        (F.col('rnk_frq') < 0.5),'Churn Risk')
                            .otherwise('NA'))
                    )
df_segment_final.display()

Churn Indicators

In [0]:
latestOrderWindow = Window.partitionBy(F.col('USER_ID')).orderBy(F.col('CREATION_TIME_UTC').desc())
df_segment = (df_segment.withColumn('cust_rnk_order',
                      F.row_number().over(latestOrderWindow))
              .filter(F.col('cust_rnk_order')==1)
              .drop('cust_rnk_order')
)
df_segment.display()